# 02 · Altair: a grammar of interaction

**Goal:** build charts declaratively, then add *selections* that link charts together: brushing, legend filtering, and widgets bound to parameters.

Plotly gives you a fixed set of interactions on each chart. Altair (built on Vega-Lite) lets you **declare** interactions: "this brush filters that chart", "this slider sets the year".

In [ ]:
import altair as alt
import pandas as pd
import plotly.express as px   # only for its bundled datasets

gap = px.data.gapminder()
g07 = gap.query("year == 2007")

## 1. The grammar: data → mark → encoding

Every Altair chart is `alt.Chart(data).mark_<type>().encode(channel="column:type")`.

| Type code | Meaning | Example |
| --- | --- | --- |
| `:Q` | quantitative | `lifeExp:Q` |
| `:N` | nominal (unordered categories) | `continent:N` |
| `:O` | ordinal (ordered categories) | `year:O` |
| `:T` | temporal | `date:T` |

In [ ]:
alt.Chart(g07).mark_circle().encode(
    x=alt.X("gdpPercap:Q").scale(type="log").title("GDP per capita (US$)"),
    y=alt.Y("lifeExp:Q").scale(zero=False).title("Life expectancy (years)"),
    size="pop:Q",
    color="continent:N",
)

## 2. The first two interactions: tooltips and `.interactive()`

Tooltips are just another encoding channel. `.interactive()` adds zoom (scroll) and pan (drag).

In [ ]:
base = alt.Chart(g07).mark_circle(opacity=0.7).encode(
    x=alt.X("gdpPercap:Q").scale(type="log").title("GDP per capita (US$)"),
    y=alt.Y("lifeExp:Q").scale(zero=False).title("Life expectancy (years)"),
    size=alt.Size("pop:Q").legend(None),
    color="continent:N",
    tooltip=["country", 
             alt.Tooltip("pop:Q", format=","),
             alt.Tooltip("gdpPercap:Q", format="$,.0f"), 
             "lifeExp"],
)
base.interactive()

## 3. Selections: interactions you declare

A **selection parameter** captures what the user did. You then use it in three ways:

1. **Highlight**: change an encoding with `alt.when(selection).then(...).otherwise(...)`
2. **Filter**: `transform_filter(selection)` on *another* chart
3. **Zoom**: use it as another chart's scale domain

### 3a. Brush to highlight

In [ ]:
brush = alt.selection_interval()   # drag a rectangle

base.encode(
    color=alt.when(brush).then("continent:N").otherwise(alt.value("lightgray"))
).add_params(brush)

### 3b. Linked views: the brush filters a second chart

Charts combine with `|` (side by side), `&` (stacked) and `+` (layered). Drag on the left;
the bar chart counts only the selected countries.

In [ ]:
scatter = base.encode(
    color=alt.when(brush).then("continent:N").otherwise(alt.value("lightgray"))
).add_params(brush).properties(width=380, height=300)

bars = alt.Chart(g07).mark_bar().encode(
    x=alt.X("count():Q").title("Countries selected"),
    y=alt.Y("continent:N").title(None),
    color="continent:N",
).transform_filter(brush).properties(width=200, height=300)

scatter | bars

### 3c. Click the legend to filter

`selection_point(fields=[...], bind="legend")` makes the legend clickable. Shift-click to pick several.

In [ ]:
pick = alt.selection_point(fields=["continent"], bind="legend")

base.encode(
    opacity=alt.when(pick).then(alt.value(0.9)).otherwise(alt.value(0.05))
).add_params(pick)

### 3d. Widgets bound to parameters

A parameter can be **bound** to an HTML widget: a slider, dropdown, radio button or checkbox. Here a slider picks the year, so we use the full dataset and filter inside the chart.

In [ ]:
year = alt.param(name="year", 
                 value=2007,
                 bind=alt.binding_range(min=1952, max=2007, step=5, name="Year "))
continent = alt.param(name="continent", 
                      value="All",
                      bind=alt.binding_select(options=["All"] + sorted(gap.continent.unique()),
                                              name="Continent "))

alt.Chart(gap).mark_circle(opacity=0.7).encode(
    x=alt.X("gdpPercap:Q").scale(type="log", domain=[200, 60_000]).title("GDP per capita"),
    y=alt.Y("lifeExp:Q").scale(domain=[25, 90]).title("Life expectancy"),
    size=alt.Size("pop:Q").scale(range=[10, 2000]).legend(None),
    color="continent:N",
    tooltip=["country", "year", "lifeExp"],
).add_params(year, continent).transform_filter(
    alt.datum.year == year
).transform_filter(
    (continent == "All") | (alt.datum.continent == continent)
)

### 3e. Overview + detail

A brush on a small overview chart sets the x-axis **domain** of the main chart: a classic
pattern for long time series.

In [ ]:
stocks = px.data.stocks().melt(id_vars="date", var_name="ticker", value_name="price")

span = alt.selection_interval(encodings=["x"])

detail = alt.Chart(stocks).mark_line().encode(
    x=alt.X("date:T").scale(domain=span).title(None),
    y=alt.Y("price:Q").title("Price relative to 1 Jan 2018"),
    color="ticker:N",
).properties(width=600, height=260)

overview = alt.Chart(stocks).mark_line().encode(
    x="date:T", y=alt.Y("price:Q").title(None), color="ticker:N",
).add_params(span).properties(width=600, height=70)

detail & overview

## 4. Things to know

- **Row limit.** Altair embeds the data in the chart spec and refuses DataFrames over 5,000 rows by default (`MaxRowsError`). Options: aggregate first, call `alt.data_transformers.disable_max_rows()` (and accept a big notebook), or install `vegafusion` and run `alt.data_transformers.enable("vegafusion")` to pre-aggregate in Python.
- **Everything runs in the browser.** Selections and filters are compiled to JavaScript, which is why these charts keep working when saved to a standalone HTML file (notebook 05).
- **Getting selections back into Python** needs a live kernel: `alt.JupyterChart` exposes selections as Python values, and Streamlit's `st.altair_chart(..., on_select="rerun")` does the same in an app.

## 5. Plotly or Altair?

| | Plotly Express | Altair |
| --- | --- | --- |
| Style | One function per chart type | Grammar: data, mark, encoding |
| Free interactions | Hover, zoom, legend toggle, PNG download | Tooltips, `.interactive()` zoom/pan |
| Linked views | Needs Dash/Streamlit callbacks | Built in (selections) |
| Widgets inside the chart | Animation slider, range slider, `updatemenus` | Any param bound to a widget |
| Large data | WebGL to ~1M points | ~5k rows default; VegaFusion helps |
| Maps, 3D | Yes | Maps yes, 3D no |

---

### ✏️ Exercise: Tips, the Altair way

Using `tips` (below), make a scatter of `total_bill` vs `tip` with:

1. color by `day`,
2. a tooltip showing `total_bill`, `tip`, `size` and `time`,
3. zoom and pan.

In [ ]:
tips = px.data.tips()
tips["tip_pct"] = tips.tip / tips.total_bill * 100

In [ ]:
# Your code here

### ✏️ Exercise: Linked brushing

Put the scatter from the previous exercise next to a **histogram of `tip_pct`**
(`mark_bar`, `x=alt.X("tip_pct:Q").bin(maxbins=20)`, `y="count()"`).
Add an interval brush to the scatter so that the histogram shows only the brushed bills.

*Bonus:* layer a gray histogram of all bills behind the filtered one so readers can compare.

In [ ]:
# Your code here

### ✏️ Exercise: A slider for party size

Add a slider (`alt.binding_range`) for a parameter `min_size` from 1 to 6, and filter the
scatter to bills with `size >= min_size`. Does the tip-to-bill relationship change for big parties?

In [ ]:
# Your code here